# Ensemble V2: XGBoost (+H2H) + Poisson

Upgrades the production ensemble by adding **Head-to-Head (H2H)** features to XGBoost.
The Poisson model is reused as-is — it does not use H2H features.

**New features vs V1:**
- `h2h_home_win_rate` — fraction of H2H meetings won by current home team (last 10)
- `h2h_avg_goal_diff` — average goal diff (home − away) from home team perspective

**Saves:**
- `models/xgb_model_v2.pkl`
- `models/ensemble_v2_meta.pkl` (metadata: feature_cols, fallback values)

**Result from experiment (notebook 04):**  
XGB v5 (+H2H): val LL=1.0128, test LL=0.9742 vs baseline val=1.0189, test=0.9843

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score

ROOT = Path('..').resolve()

COMPETITIVE = [
    'FIFA World Cup', 'FIFA World Cup qualification',
    'UEFA Euro', 'UEFA Euro qualification',
    'Copa América', 'African Cup of Nations',
    'AFC Asian Cup', 'Gold Cup',
    'UEFA Nations League', 'CONCACAF Nations League',
]

RESULT_MAP     = {'home_win': 0, 'draw': 1, 'away_win': 2}
RESULT_MAP_INV = {0: 'home_win', 1: 'draw', 2: 'away_win'}

FEATURE_COLS_V2 = [
    'home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded',
    'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded',
    'elo_diff', 'neutral', 'tournament',
    'h2h_home_win_rate', 'h2h_avg_goal_diff',
]

features = pd.read_csv(ROOT / 'data/processed/features_v5.csv', parse_dates=['date'])
print('features_v5 shape:', features.shape)
print('columns:', list(features.columns))

features_v5 shape: (16610, 19)
columns: ['date', 'home_team', 'away_team', 'neutral', 'tournament', 'home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded', 'elo_diff', 'home_elo', 'away_elo', 'result', 'h2h_home_win_rate', 'h2h_avg_goal_diff']


In [2]:
# Same time-based split as production v1
train = features[features['date'] < '2022-11-20'].copy()
val   = features[
    (features['date'] >= '2022-11-20') &
    (features['date'] <= '2022-12-18') &
    (features['tournament'] == 'FIFA World Cup')
].copy()
test  = features[features['date'] >= '2026-06-11'].copy()

print(f'Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')

le = LabelEncoder()
le.fit(features['tournament'])

for df in (train, val, test):
    df['neutral']    = df['neutral'].astype(int)
    df['tournament'] = le.transform(df['tournament'])
    df['label']      = df['result'].map(RESULT_MAP)

Train: 14639, Val: 64, Test: 28


In [3]:
xgb_v2 = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    n_estimators=300, learning_rate=0.05,
    max_depth=4, subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    early_stopping_rounds=20, random_state=42,
)
xgb_v2.fit(
    train[FEATURE_COLS_V2], train['label'],
    eval_set=[(val[FEATURE_COLS_V2], val['label'])],
    verbose=False,
)
print(f'XGBoost v2 trained. Best iteration: {xgb_v2.best_iteration}')

p_xgb_val  = xgb_v2.predict_proba(val[FEATURE_COLS_V2])
p_xgb_test = xgb_v2.predict_proba(test[FEATURE_COLS_V2])
print(f'XGB v2  val LL: {log_loss(val["label"], p_xgb_val):.4f}')
print(f'XGB v2 test LL: {log_loss(test["label"], p_xgb_test):.4f}')

/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/xgboost/callback.py:385: UserWarning: [11:40:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


XGBoost v2 trained. Best iteration: 90
XGB v2  val LL: 1.0128
XGB v2 test LL: 0.9742


In [4]:
# Load existing Poisson model (unchanged — does not use H2H)
with open(ROOT / 'models/poisson_model.pkl', 'rb') as f:
    poi = pickle.load(f)

from scipy.stats import poisson as scipy_poisson

def poisson_proba(lam_h, lam_a, max_goals=10):
    p_home = p_draw = p_away = 0.0
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            p = scipy_poisson.pmf(i, lam_h) * scipy_poisson.pmf(j, lam_a)
            if   i > j: p_home += p
            elif i==j:  p_draw += p
            else:       p_away += p
    return np.array([p_home, p_draw, p_away])

def poisson_batch(df):
    X_h = poi['scaler_home'].transform(df[poi['feats_home']])
    X_a = poi['scaler_away'].transform(df[poi['feats_away']])
    lam_h = poi['model_home'].predict(X_h)
    lam_a = poi['model_away'].predict(X_a)
    return np.vstack([poisson_proba(lh, la) for lh, la in zip(lam_h, lam_a)])

# Poisson needs original (unencoded) neutral as int — already done above
p_poi_val  = poisson_batch(val)
p_poi_test = poisson_batch(test)
print(f'Poisson  val LL: {log_loss(val["label"], p_poi_val):.4f}')
print(f'Poisson test LL: {log_loss(test["label"], p_poi_test):.4f}')

Poisson  val LL: 1.0307
Poisson test LL: 0.9798


/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [5]:
# Ensemble V2: simple average XGB_v2 + Poisson
p_ens_val  = (p_xgb_val  + p_poi_val)  / 2
p_ens_test = (p_xgb_test + p_poi_test) / 2

# Load V1 XGB for comparison
with open(ROOT / 'models/xgb_model.pkl', 'rb') as f:
    xgb_v1_art = pickle.load(f)

FEATURE_COLS_V1 = xgb_v1_art['feature_cols']
le_v1 = xgb_v1_art['le_tournament']

val_v1  = features[
    (features['date'] >= '2022-11-20') &
    (features['date'] <= '2022-12-18') &
    (features['tournament'] == 0)  # already encoded above
].copy()

# Re-encode for v1 (different encoder)
feat_orig = pd.read_csv(ROOT / 'data/processed/features_v5.csv', parse_dates=['date'])
val_orig  = feat_orig[
    (feat_orig['date'] >= '2022-11-20') &
    (feat_orig['date'] <= '2022-12-18') &
    (feat_orig['tournament'] == 'FIFA World Cup')
].copy()
test_orig = feat_orig[feat_orig['date'] >= '2026-06-11'].copy()

for df in (val_orig, test_orig):
    df['neutral']    = df['neutral'].astype(int)
    df['tournament'] = le_v1.transform(df['tournament'])
    df['label']      = df['result'].map(RESULT_MAP)

p_xgb_v1_val  = xgb_v1_art['model'].predict_proba(val_orig[FEATURE_COLS_V1])
p_xgb_v1_test = xgb_v1_art['model'].predict_proba(test_orig[FEATURE_COLS_V1])
p_poi_val_v1  = poisson_batch(val_orig)
p_poi_test_v1 = poisson_batch(test_orig)
p_ens_v1_val  = (p_xgb_v1_val  + p_poi_val_v1)  / 2
p_ens_v1_test = (p_xgb_v1_test + p_poi_test_v1) / 2

y_val  = val_orig['label']
y_test = test_orig['label']

print('='*65)
print(f'{"Model":<30} {"Val LL":>10} {"Test LL":>10} {"Test Acc":>10}')
print('-'*65)
rows = [
    ('XGB v1 (baseline)',        p_xgb_v1_val,  p_xgb_v1_test),
    ('XGB v2 (+H2H)',            p_xgb_val,     p_xgb_test),
    ('Poisson (unchanged)',      p_poi_val_v1,  p_poi_test_v1),
    ('Ensemble V1 (prod)',       p_ens_v1_val,  p_ens_v1_test),
    ('Ensemble V2 (XGB_v2+Poi)', p_ens_val,     p_ens_test),
]
for name, pv, pt in rows:
    vl  = log_loss(y_val,  pv)
    tl  = log_loss(y_test, pt)
    acc = accuracy_score(y_test, pt.argmax(axis=1))
    print(f'{name:<30} {vl:>10.4f} {tl:>10.4f} {acc:>10.1%}')
print('='*65)

Model                              Val LL    Test LL   Test Acc
-----------------------------------------------------------------
XGB v1 (baseline)                  1.0189     0.9843      53.6%
XGB v2 (+H2H)                      1.0128     0.9742      53.6%
Poisson (unchanged)                1.0307     0.9798      50.0%
Ensemble V1 (prod)                 1.0188     0.9796      50.0%
Ensemble V2 (XGB_v2+Poi)           1.0151     0.9747      50.0%


/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to p

In [6]:
# Save XGBoost v2 artifact
xgb_v2_artifact = {
    'model':          xgb_v2,
    'le_tournament':  le,
    'feature_cols':   FEATURE_COLS_V2,
    'result_map':     RESULT_MAP,
    'result_map_inv': RESULT_MAP_INV,
    # H2H fallback values (for predict.py when no H2H history)
    'h2h_fallback': {'h2h_home_win_rate': 0.5, 'h2h_avg_goal_diff': 0.0},
}

with open(ROOT / 'models/xgb_model_v2.pkl', 'wb') as f:
    pickle.dump(xgb_v2_artifact, f)

print('Saved models/xgb_model_v2.pkl')
print(f'  feature_cols ({len(FEATURE_COLS_V2)}): {FEATURE_COLS_V2}')
print(f'  best_iteration: {xgb_v2.best_iteration}')
print(f'  h2h_fallback: {xgb_v2_artifact["h2h_fallback"]}')

Saved models/xgb_model_v2.pkl
  feature_cols (13): ['home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded', 'elo_diff', 'neutral', 'tournament', 'h2h_home_win_rate', 'h2h_avg_goal_diff']
  best_iteration: 90
  h2h_fallback: {'h2h_home_win_rate': 0.5, 'h2h_avg_goal_diff': 0.0}


In [7]:
# Verify artifact round-trip
with open(ROOT / 'models/xgb_model_v2.pkl', 'rb') as f:
    loaded = pickle.load(f)

p_check = loaded['model'].predict_proba(val[FEATURE_COLS_V2])
assert np.allclose(p_check, p_xgb_val, atol=1e-5), 'Round-trip mismatch!'
print('Round-trip check: OK')
print(f'Keys: {list(loaded.keys())}')
print(f'Feature cols: {loaded["feature_cols"]}')

Round-trip check: OK
Keys: ['model', 'le_tournament', 'feature_cols', 'result_map', 'result_map_inv', 'h2h_fallback']
Feature cols: ['home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded', 'elo_diff', 'neutral', 'tournament', 'h2h_home_win_rate', 'h2h_avg_goal_diff']


---
## Poisson V2: adding h2h_avg_goal_diff

**Hypothesis:** `h2h_avg_goal_diff` (average goal difference in historical head-to-head matches)
is directly related to goals scored — exactly what Poisson predicts.

Added to both feature sets:
- `FEATS_HOME` (+h2h_avg_goal_diff) → predicts λ_home
- `FEATS_AWAY` (+h2h_avg_goal_diff) → predicts λ_away

The model learns the correct sign automatically:
- λ_home: positive h2h_diff → home team historically scores more
- λ_away: positive h2h_diff → away team historically scores fewer goals (negative relationship)

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor

FEATS_HOME_V2 = [
    'home_avg_goals_scored', 'away_avg_goals_conceded',
    'home_win_rate', 'elo_diff', 'neutral',
    'h2h_avg_goal_diff',
]
FEATS_AWAY_V2 = [
    'away_avg_goals_scored', 'home_avg_goals_conceded',
    'away_win_rate', 'elo_diff', 'neutral',
    'h2h_avg_goal_diff',
]

# Load actual scores for Poisson training (need home_score, away_score)
results_raw = pd.read_csv(ROOT / 'data/processed/results_historical.csv', parse_dates=['date'])
features_v5_raw = pd.read_csv(ROOT / 'data/processed/features_v5.csv', parse_dates=['date'])

# Merge scores into features_v5
df_scored = features_v5_raw.merge(
    results_raw[['date','home_team','away_team','home_score','away_score']],
    on=['date','home_team','away_team'], how='inner'
)
print(f'Merged shape: {df_scored.shape}')

# Same train split (no val/test needed for Poisson training — we use full pre-2022 train)
train_p = df_scored[df_scored['date'] < '2022-11-20'].copy()
train_p['neutral'] = train_p['neutral'].astype(int)
print(f'Poisson train: {len(train_p)}')

scaler_home_v2 = StandardScaler()
scaler_away_v2 = StandardScaler()

X_train_h = scaler_home_v2.fit_transform(train_p[FEATS_HOME_V2])
X_train_a = scaler_away_v2.fit_transform(train_p[FEATS_AWAY_V2])

poi_home_v2 = PoissonRegressor(alpha=0.01, max_iter=1000)
poi_away_v2 = PoissonRegressor(alpha=0.01, max_iter=1000)
poi_home_v2.fit(X_train_h, train_p['home_score'])
poi_away_v2.fit(X_train_a, train_p['away_score'])

# Check λ range on train (must have variance, not collapsed)
lam_h = poi_home_v2.predict(X_train_h)
lam_a = poi_away_v2.predict(X_train_a)
print(f'λ_home range: [{lam_h.min():.2f}, {lam_h.max():.2f}]  std={lam_h.std():.3f}')
print(f'λ_away range: [{lam_a.min():.2f}, {lam_a.max():.2f}]  std={lam_a.std():.3f}')

# Coefficients — check h2h_avg_goal_diff sign
print('\nh2h_avg_goal_diff coefficient:')
print(f'  model_home: {poi_home_v2.coef_[FEATS_HOME_V2.index("h2h_avg_goal_diff")]:+.4f} (expect positive)')
print(f'  model_away: {poi_away_v2.coef_[FEATS_AWAY_V2.index("h2h_avg_goal_diff")]:+.4f} (expect negative)')

Merged shape: (16610, 21)
Poisson train: 14639
λ_home range: [0.31, 60.91]  std=1.045
λ_away range: [0.17, 37.89]  std=0.718

h2h_avg_goal_diff coefficient:
  model_home: +0.0934 (expect positive)
  model_away: -0.1087 (expect negative)


In [11]:
# Evaluate Poisson v2 on val and test
def poisson_batch_v2(df):
    df = df.copy()
    df['neutral'] = df['neutral'].astype(int) if df['neutral'].dtype == bool else df['neutral']
    X_h = scaler_home_v2.transform(df[FEATS_HOME_V2])
    X_a = scaler_away_v2.transform(df[FEATS_AWAY_V2])
    lam_h = poi_home_v2.predict(X_h)
    lam_a = poi_away_v2.predict(X_a)
    return np.vstack([poisson_proba(lh, la) for lh, la in zip(lam_h, lam_a)])

# Load val/test from features_v5 (needs h2h_avg_goal_diff column)
feat_raw = pd.read_csv(ROOT / 'data/processed/features_v5.csv', parse_dates=['date'])
val_raw  = feat_raw[
    (feat_raw['date'] >= '2022-11-20') &
    (feat_raw['date'] <= '2022-12-18') &
    (feat_raw['tournament'] == 'FIFA World Cup')
].copy()
test_raw = feat_raw[feat_raw['date'] >= '2026-06-11'].copy()
for df_ in (val_raw, test_raw):
    df_['neutral'] = df_['neutral'].astype(int)

p_poi2_val  = poisson_batch_v2(val_raw)
p_poi2_test = poisson_batch_v2(test_raw)
print(f'val shapes  — p_poi2: {p_poi2_val.shape},  y_val: {len(y_val)}')
print(f'test shapes — p_poi2: {p_poi2_test.shape}, y_test: {len(y_test)}')

# Ensemble V2b: XGB v2 + Poisson v2
p_ens2b_val  = (p_xgb_val  + p_poi2_val)  / 2
p_ens2b_test = (p_xgb_test + p_poi2_test) / 2

def acc(y, p):
    return f'{accuracy_score(y, p.argmax(axis=1)):.1%}'

print('='*68)
print(f'{"Model":<34} {"Val LL":>8} {"Test LL":>9} {"Test Acc":>10}')
print('-'*68)
rows = [
    ('Ensemble V1 (prod)',            1.0188,                         0.9796,                         '50.0%'),
    ('XGB v2 alone (+H2H)',           log_loss(y_val, p_xgb_val),    log_loss(y_test, p_xgb_test),   acc(y_test, p_xgb_test)),
    ('Poisson v1 (original)',         log_loss(y_val, p_poi_val),    log_loss(y_test, p_poi_test),   acc(y_test, p_poi_test)),
    ('Poisson v2 (+h2h_goal_diff)',   log_loss(y_val, p_poi2_val),   log_loss(y_test, p_poi2_test),  acc(y_test, p_poi2_test)),
    ('Ensemble V2  (XGB_v2+Poi_v1)', log_loss(y_val, p_ens_val),    log_loss(y_test, p_ens_test),   acc(y_test, p_ens_test)),
    ('Ensemble V2b (XGB_v2+Poi_v2)', log_loss(y_val, p_ens2b_val),  log_loss(y_test, p_ens2b_test), acc(y_test, p_ens2b_test)),
]
for name, vl, tl, acc_str in rows:
    print(f'{name:<34} {vl:>8.4f} {tl:>9.4f} {acc_str:>10}')
print('='*68)


val shapes  — p_poi2: (64, 3),  y_val: 64
test shapes — p_poi2: (28, 3), y_test: 28
Model                                Val LL   Test LL   Test Acc
--------------------------------------------------------------------
Ensemble V1 (prod)                   1.0188    0.9796      50.0%
XGB v2 alone (+H2H)                  1.0128    0.9742      53.6%
Poisson v1 (original)                1.0307    0.9798      50.0%
Poisson v2 (+h2h_goal_diff)          1.0293    0.9726      50.0%
Ensemble V2  (XGB_v2+Poi_v1)         1.0151    0.9747      50.0%
Ensemble V2b (XGB_v2+Poi_v2)         1.0167    0.9717      50.0%


/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/Users/bony-macpro/Documents/BeCode/Projects/World-Cup-Match-Prediction/.worldcupfootball/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to p

In [12]:
# Save Poisson v2 artifact
poisson_v2_artifact = {
    'model_home':   poi_home_v2,
    'model_away':   poi_away_v2,
    'scaler_home':  scaler_home_v2,
    'scaler_away':  scaler_away_v2,
    'feats_home':   FEATS_HOME_V2,
    'feats_away':   FEATS_AWAY_V2,
    'result_map':     RESULT_MAP,
    'result_map_inv': RESULT_MAP_INV,
}

with open(ROOT / 'models/poisson_model_v2.pkl', 'wb') as f:
    pickle.dump(poisson_v2_artifact, f)

print('Saved models/poisson_model_v2.pkl')
print(f'  feats_home ({len(FEATS_HOME_V2)}): {FEATS_HOME_V2}')
print(f'  feats_away ({len(FEATS_AWAY_V2)}): {FEATS_AWAY_V2}')

# Verify round-trip
with open(ROOT / 'models/poisson_model_v2.pkl', 'rb') as f:
    poi2_check = pickle.load(f)
X_check = poi2_check['scaler_home'].transform(val_raw[FEATS_HOME_V2])
lam_check = poi2_check['model_home'].predict(X_check)
print(f'Round-trip check: λ_home[0]={lam_check[0]:.4f} — OK')

Saved models/poisson_model_v2.pkl
  feats_home (6): ['home_avg_goals_scored', 'away_avg_goals_conceded', 'home_win_rate', 'elo_diff', 'neutral', 'h2h_avg_goal_diff']
  feats_away (6): ['away_avg_goals_scored', 'home_avg_goals_conceded', 'away_win_rate', 'elo_diff', 'neutral', 'h2h_avg_goal_diff']
Round-trip check: λ_home[0]=1.6280 — OK
